In [1]:
import polaris as po
import pandas as pd

/home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the dataset from the Hub
dataset = po.load_dataset("asap-discovery/antiviral-potency-2025-unblinded")


/home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/rich/live.
py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[2025-11-28 10:58:20] INFO     The version of Polaris that was used to create the artifact          ]8;id=28856;file:///home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/polaris/_artifact.py\_artifact.py]8;;\:]8;id=984264;file:///home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/polaris/_artifact.py#96\96]8;;\
                               (0.11.8.dev4+g40e3b2b.d20250207) is different from the currently                    
                               installed version of Polaris (0.13.0).                                              

                      WARNING  You're loading data from a remote location. If the dataset is small     ]8;id=962163;file:///home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/polaris/dataset/_base.py\_base.py]8;;\:]8;id=421179;file:///home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/polaris/dataset/_base.py#187\187]8;;\
                               enough, consider caching the dataset first using DatasetV2.cache() for              
                               more performant data access.                                                        

[10:58:21]  Success: Fetching dataset                                                                 ]8;id=679496;file:///home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/polaris/utils/context.py\context.py]8;;\:]8;id=407652;file:///home/jannik/Documents/studies/phd/03_work/20_FragShapley/FragShapley/.venv/lib/python3.12/site-packages/polaris/utils/context.py#53\53]8;;\

In [3]:
# Get information on the dataset size
dataset.size()

# Load a datapoint in memory
dataset.get_data(
    row=dataset.rows[0],
    col=dataset.columns[0],
)

'COC[C@]1(C)C(=O)N(C2=CN=CC3=CC=CC=C23)C(=O)N1C |&1:3|'

In [4]:
df = pd.DataFrame(dataset[:])
df.head()

,CXSMILES,Molecule Name,Set,pIC50 (MERS-CoV Mpro),pIC50 (SARS-CoV-2 Mpro)
0,COC[C@]1(C)C(=O)N(C2=CN=CC3=CC=CC=C23)C(=O)N1C...,ASAP-0000141,Train,4.19,NaN
1,C=C(CN1CCC2=C(C=C(Cl)C=C2)C1C(=O)NC1=CN=CC2=CC...,ASAP-0000142,Train,4.92,5.29
2,CNC(=O)CN1C[C@]2(C[C@H](C)N(C3=CN=CC=C3C3CC3)C...,ASAP-0000143,Train,4.73,NaN
3,C=C(CN1CCC2=C(C=C(Cl)C=C2)C1C(=O)NC1=CN=CC2=CC...,ASAP-0000144,Train,4.90,6.11
4,C=C(CN1CCC2=C(C=C(Cl)C=C2)C1C(=O)NC1=CN=CC2=CC...,ASAP-0000145,Train,4.81,5.62


In [5]:
# check if we can turn all SMILES into valid rdkit molecules

In [6]:
# generate nonstereo_aromatic_smiles
from rdkit.Chem import MolFromSmiles, MolToSmiles

def get_canonical_SMILES(smiles):
    try:
        mol = MolFromSmiles(smiles)
    except:
        return False
    if mol is None:
        return False
    return MolToSmiles(mol, canonical=True)

df['canonical_smiles'] = df.CXSMILES.apply(get_canonical_SMILES)

# unmodified dataset

In [7]:
for ep, eps in zip(['pIC50 (MERS-CoV Mpro)', 'pIC50 (SARS-CoV-2 Mpro)'], ["MERS", "SARS"]):
    tmp = df[['canonical_smiles', ep, "Set"]]
    tmp = tmp.rename(columns={ep: "pIC50", "Set": "set"})
    tmp = tmp.dropna(subset="pIC50")
    tmp.to_csv(f'regression/{eps}.csv', index=False)


# modify datasets

Remove all entries that are duplicate. This is due to improper designation of the chirality of some molecules.

In [8]:
# get list of all canoncial_smiles which are duplicated smiles
for ep, eps in zip(['pIC50 (MERS-CoV Mpro)', 'pIC50 (SARS-CoV-2 Mpro)'], ["MERS", "SARS"]):
    tmp = df[['canonical_smiles', ep, "Set"]]
    tmp = tmp.rename(columns={ep: "pIC50", "Set": "set"})
    tmp = tmp.dropna(subset="pIC50")
    dup_smiles = tmp.loc[tmp.canonical_smiles.duplicated()].canonical_smiles.to_list()
    tmp = tmp.query('canonical_smiles not in @dup_smiles')
    tmp.to_csv(f'regression/{eps}_mod.csv', index=False)